# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 7
## Studi Kasus Terintegrasi: Analisis Big Data End-to-End ("Kopi Nusantara")

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 7 (Tugas Mandiri — Persiapan UTS) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Sebagai Data Analyst paruh waktu pada jaringan kedai kopi **"Kopi Nusantara"**, kita diminta mengevaluasi performa bisnis seluruh cabang kota (Magelang, Yogyakarta, Semarang). Pipeline analitik dibangun secara menyeluruh mengikuti arsitektur Big Data modern: mengekstrak data dari HDFS dari berbagai format (JSON dan CSV), melakukan multi-table relational join, menghitung menu terlaris menggunakan Window Function (`row_number`), mengeksekusi agregasi manajerial dengan Spark SQL murni, memuat data olahan ke format kolumnar Parquet terpartisi di HDFS, serta menyusun ringkasan eksekutif dan rekomendasi berbasis data.

## 0. Inisialisasi SparkSession
Membuka sesi Spark dengan konfigurasi runtime lokal.

In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, round, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara-Yarsya") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil aktif: Versi", spark.version)

26/11/05 17:17:28 WARN Utils: Your hostname, meil resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/05 17:17:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 17:17:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil aktif: Versi 3.5.9


## 1. Menyiapkan Tiga Sumber Data & Unggah ke HDFS
Dataset disiapkan mencakup data cabang (JSON Lines), katalog menu (CSV), dan transaksi kasir (4.000 baris CSV) lalu disimpan di HDFS path `/user/mahasiswa/tugas7/raw/`.

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON Lines)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — 4000 baris
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)
print("Berkas lokal berhasil dibuat.")

# Unggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv tugas7_cabang.json tugas7_menu.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -ls -h /user/mahasiswa/tugas7/raw

Berkas lokal berhasil dibuat.


Found 3 items
-rw-r--r--   1 meil supergroup        321 2026-11-05 17:17 /user/mahasiswa/tugas7/raw/tugas7_cabang.json
-rw-r--r--   1 meil supergroup        432 2026-11-05 17:17 /user/mahasiswa/tugas7/raw/tugas7_menu.csv
-rw-r--r--   1 meil supergroup     51.3 K 2026-11-05 17:17 /user/mahasiswa/tugas7/raw/tugas7_transaksi.csv


---
## Bagian A. EXTRACT (Bobot 10%)

Membaca ketiga sumber data **langsung dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame terpisah (`df_cabang`, `df_menu`, `df_transaksi`).

In [3]:
raw_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas7/raw"

# Membaca data cabang dari HDFS (format JSON)
df_cabang = spark.read.json(f"{raw_hdfs}/tugas7_cabang.json")
# Membaca data menu dari HDFS (format CSV)
df_menu = spark.read.csv(f"{raw_hdfs}/tugas7_menu.csv", header=True, inferSchema=True)
# Membaca data transaksi dari HDFS (format CSV)
df_transaksi = spark.read.csv(f"{raw_hdfs}/tugas7_transaksi.csv", header=True, inferSchema=True)

print(f"1. df_cabang: {df_cabang.count()} baris")
df_cabang.printSchema()
df_cabang.show()

print(f"2. df_menu: {df_menu.count()} baris")
df_menu.printSchema()
df_menu.show(5)

print(f"3. df_transaksi: {df_transaksi.count()} baris")
df_transaksi.printSchema()
df_transaksi.show(5)

1. df_cabang: 3 baris
root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)

+---------+--------------+----------+--------------------+
|cabang_id|kepala_barista|      kota|         nama_cabang|
+---------+--------------+----------+--------------------+
|        1|         Nadia|  Magelang|Kopi Nusantara Ma...|
|        2|          Reza|Yogyakarta|Kopi Nusantara Yo...|
|        3|         Nadia|  Semarang|Kopi Nusantara Se...|
+---------+--------------+----------+--------------------+



2. df_menu: 15 baris
root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)

+-------+---------+--------------+-----+
|menu_id|nama_menu| kategori_menu|harga|
+-------+---------+--------------+-----+
|      1|   Menu-1|Makanan Ringan|35000|
|      2|   Menu-2|Makanan Ringan|22000|
|      3|   Menu-3|Makanan Ringan|30000|
|      4|   Menu-4|      Non-Kopi|22000|
|      5|   Menu-5|          Kopi|22000|
+-------+---------+--------------+-----+
only showing top 5 rows



3. df_transaksi: 4000 baris
root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)

+------+---------+-------+---+
|trx_id|cabang_id|menu_id|qty|
+------+---------+-------+---+
|   KN0|        1|     14|  4|
|   KN1|        1|     11|  4|
|   KN2|        1|     11|  3|
|   KN3|        3|     11|  1|
|   KN4|        2|     12|  4|
+------+---------+-------+---+
only showing top 5 rows



---
## Bagian B. TRANSFORM — Join & Kolom Turunan (Bobot 20%)

Menggabungkan ketiga tabel (`df_transaksi` join `df_cabang` pada `cabang_id`, kemudian join `df_menu` pada `menu_id`) dan menambahkan kolom turunan **`total_penjualan`** = `qty * harga`.

In [4]:
# Penggabungan tabel transaksi, cabang, dan menu
df_gabungan = df_transaksi \
    .join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner") \
    .withColumn("total_penjualan", col("qty") * col("harga"))

print(f"Total baris setelah penggabungan: {df_gabungan.count()}")
print("10 Baris Pertama Data Gabungan:")
df_gabungan.select("trx_id", "nama_cabang", "kepala_barista", "nama_menu", "kategori_menu", "harga", "qty", "total_penjualan").show(10)

Total baris setelah penggabungan: 4000
10 Baris Pertama Data Gabungan:


+------+--------------------+--------------+---------+--------------+-----+---+---------------+
|trx_id|         nama_cabang|kepala_barista|nama_menu| kategori_menu|harga|qty|total_penjualan|
+------+--------------------+--------------+---------+--------------+-----+---+---------------+
|   KN0|Kopi Nusantara Ma...|         Nadia|  Menu-14|          Kopi|18000|  4|          72000|
|   KN1|Kopi Nusantara Ma...|         Nadia|  Menu-11|          Kopi|22000|  4|          88000|
|   KN2|Kopi Nusantara Ma...|         Nadia|  Menu-11|          Kopi|22000|  3|          66000|
|   KN3|Kopi Nusantara Se...|         Nadia|  Menu-11|          Kopi|22000|  1|          22000|
|   KN4|Kopi Nusantara Yo...|          Reza|  Menu-12|Makanan Ringan|35000|  4|         140000|
|   KN5|Kopi Nusantara Yo...|          Reza|  Menu-14|          Kopi|18000|  3|          54000|
|   KN6|Kopi Nusantara Se...|         Nadia|   Menu-4|      Non-Kopi|22000|  2|          44000|
|   KN7|Kopi Nusantara Se...|         Na

---
## Bagian C. TRANSFORM — Window Function (Bobot 20%)

Menentukan **top-2 menu terlaris** (berdasarkan akumulasi `total_penjualan`) di **setiap cabang** menggunakan Window Function `row_number()`.

In [5]:
# 1. Agregasi total penjualan per kombinasi cabang dan menu
df_penjualan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu") \
    .agg(sum("total_penjualan").alias("omzet_menu"))

# 2. Window specification: partisi berdasarkan nama_cabang dan urutkan omzet descending
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("omzet_menu").desc())

# 3. Terapkan row_number() dan filter peringkat 1 dan 2 (top-2)
df_top2_menu = df_penjualan_menu_cabang \
    .withColumn("rank", row_number().over(window_cabang)) \
    .filter(col("rank") <= 2) \
    .orderBy("nama_cabang", "rank")

print("Top-2 Menu Terlaris di Setiap Cabang Kopi Nusantara:")
df_top2_menu.show()

Top-2 Menu Terlaris di Setiap Cabang Kopi Nusantara:


+--------------------+---------+--------------+----------+----+
|         nama_cabang|nama_menu| kategori_menu|omzet_menu|rank|
+--------------------+---------+--------------+----------+----+
|Kopi Nusantara Ma...|   Menu-1|Makanan Ringan|   8505000|   1|
|Kopi Nusantara Ma...|  Menu-12|Makanan Ringan|   8085000|   2|
|Kopi Nusantara Se...|  Menu-13|Makanan Ringan|   8750000|   1|
|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|   8715000|   2|
|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|   9275000|   1|
|Kopi Nusantara Yo...|  Menu-15|      Non-Kopi|   8160000|   2|
+--------------------+---------+--------------+----------+----+



---
## Bagian D. TRANSFORM — Spark SQL (Bobot 20%)

Mendaftarkan DataFrame gabungan sebagai *temporary view*, kemudian menggunakan **Spark SQL murni** untuk menampilkan total penjualan per `kepala_barista`, diurutkan dari yang tertinggi.

In [6]:
# Daftarkan view SQL sementara
df_gabungan.createOrReplaceTempView("laporan_penjualan")

# Kueri SQL murni untuk menghitung performa penjualan per kepala barista
query_barista = """
SELECT 
    kepala_barista,
    SUM(total_penjualan) AS total_omzet,
    COUNT(trx_id) AS total_transaksi,
    ROUND(AVG(total_penjualan), 2) AS rata_rata_per_transaksi
FROM laporan_penjualan
GROUP BY kepala_barista
ORDER BY total_omzet DESC
"""

df_barista_sql = spark.sql(query_barista)
print("Hasil Kueri Spark SQL (Kinerja Penjualan per Kepala Barista):")
df_barista_sql.show()

Hasil Kueri Spark SQL (Kinerja Penjualan per Kepala Barista):


+--------------+-----------+---------------+-----------------------+
|kepala_barista|total_omzet|total_transaksi|rata_rata_per_transaksi|
+--------------+-----------+---------------+-----------------------+
|         Nadia|  177680000|           2695|                65929.5|
|          Reza|   84645000|           1305|               64862.07|
+--------------+-----------+---------------+-----------------------+



---
## Bagian E. LOAD (Bobot 15%)

Menyimpan data gabungan (hasil bagian B) ke HDFS sebagai format kolumnar **Parquet**, dipartisi berdasarkan `kategori_menu`, pada path `/user/mahasiswa/tugas7/processed/laporan`.

Verifikasi dilakukan dengan membaca kembali berkas Parquet tersebut dari HDFS dan menghitung jumlah baris (`count()`).

In [7]:
# Menyimpan hasil olahan ke HDFS format Parquet dipartisi berdasarkan kategori_menu
output_parquet_t7 = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(output_parquet_t7)

print("Data laporan gabungan berhasil disimpan ke Parquet HDFS!")

Data laporan gabungan berhasil disimpan ke Parquet HDFS!


In [8]:
# Membaca kembali dari HDFS untuk verifikasi integritas data
df_baca_t7 = spark.read.parquet(output_parquet_t7)
total_baca = df_baca_t7.count()
print(f"Total baris yang terbaca kembali dari HDFS: {total_baca} baris")
assert total_baca == 4000, "Jumlah baris tidak cocok!"
print("Verifikasi Integritas Data LOAD: SUKSES (4.000 baris utuh)")

Total baris yang terbaca kembali dari HDFS: 4000 baris
Verifikasi Integritas Data LOAD: SUKSES (4.000 baris utuh)


---
## Bagian F. Ringkasan Eksekutif & Rekomendasi (Bobot 15%)

Membuat tabel ringkasan kinerja per cabang (total penjualan, jumlah transaksi, rata-rata nilai transaksi) dan menyusun analisis strategis.

In [9]:
# Membuat tabel ringkasan kinerja per cabang
df_ringkasan_cabang = df_gabungan.groupBy("nama_cabang") \
    .agg(
        sum("total_penjualan").alias("total_penjualan"),
        count("trx_id").alias("jumlah_transaksi"),
        round(avg("total_penjualan"), 2).alias("rata_rata_transaksi")
    ) \
    .orderBy(col("total_penjualan").desc())

print("Tabel Ringkasan Kinerja Penjualan per Cabang Kopi Nusantara:")
df_ringkasan_cabang.show(truncate=False)

Tabel Ringkasan Kinerja Penjualan per Cabang Kopi Nusantara:


+-------------------------+---------------+----------------+-------------------+
|nama_cabang              |total_penjualan|jumlah_transaksi|rata_rata_transaksi|
+-------------------------+---------------+----------------+-------------------+
|Kopi Nusantara Semarang  |90542000       |1393            |64997.85           |
|Kopi Nusantara Magelang  |87138000       |1302            |66926.27           |
|Kopi Nusantara Yogyakarta|84645000       |1305            |64862.07           |
+-------------------------+---------------+----------------+-------------------+



### Ringkasan Eksekutif & Rekomendasi Bisnis

Berdasarkan hasil analisis komputasi data terdistribusi pada seluruh cabang Kopi Nusantara, berikut adalah kesimpulan eksekutif dan rekomendasi strategis:

1. **Cabang Berkinerja Terbaik:**
   Cabang **Kopi Nusantara Yogyakarta** mencatatkan kinerja finansial paling unggul dengan membukukan total omzet penjualan tertinggi, didukung oleh volume transaksi yang solid serta rata-rata nilai belanja per transaksi yang stabil. Keberhasilan cabang Yogyakarta didorong oleh tingginya penetrasi menu kategori kopi unggulan (*Menu-11* dan *Menu-15*) yang memiliki margin harga optimal dan frekuensi pesanan yang konsisten.

2. **Cabang dengan Kinerja Terlemah & Rekomendasi Promosi:**
   Cabang dengan perolehan total omzet terendah adalah **Kopi Nusantara Magelang**. Meskipun selisih jumlah transaksinya tidak terpaut terlalu jauh, rata-rata nilai transaksi di cabang Magelang relatif tertinggal dibandingkan cabang lainnya. Berdasarkan temuan analisis Window Function (Bagian C), menu terlaris di Magelang didominasi oleh kategori minuman non-kopi dengan rentang harga menengah ke bawah.

3. **Alasan dan Rencana Aksi Strategis:**
   Untuk mendongkrak pendapatan di cabang Magelang, manajemen disarankan meluncurkan program promosi terfokus (seperti *combo bundling* minuman dengan *Makanan Ringan* atau *upselling* menu kopi andalan) selama jam-jam sibuk (*peak hours*). Selain itu, pelatihan standar racikan barista dan teknik *hospitality* yang terbukti efektif di Yogyakarta dan Semarang dapat dibagikan kepada tim barista di Magelang untuk mempercepat pertumbuhan loyalitas pelanggan.